# Outline
The purpose of this script is to compute spectral information for each of the inferred modes

In [ ]:
# Imports
import os
import os.path as op
from osl_dynamics.data import Data
from osl_dynamics.models import load
from osl_dynamics.analysis import spectral
from osl_dynamics.inference import modes
import pickle
import numpy as np
import glob
import yaml


In [ ]:
# Wriet out data?
writeOK=True

In [ ]:
# Read config file and define paths
with open('0_config.yaml', 'r') as file:
    config = yaml.safe_load(file)
    
subjects_dir = config['dirs']['recon_dir']
proc_dir = config['dirs']['proc_dir']
atlas_dir = config['dirs']['atlas_dir']
model_dir = config['dirs']['model_dir']
results_dir = config['dirs']['results_dir']



# Plotting params
colours = config['misc']['colours']
n_modes = 6

# Other params
run = '5'

# Update model dir according to run #
model_dir = f'{model_dir}_run{run}'

In [ ]:
# Define a helper function to pull the most recent version of a file
# Note that this sleects file based on the date & time that they were last
# modified, not the actual datetime printed in the filename 
def recent_fname(path):
    
    # Takes a partial file path
    files = glob.glob(path)
    most_recent_file = max(files, key=os.path.getmtime)
    
    return most_recent_file

In [ ]:
# Load list of subjects
subjects_fname = recent_fname(os.path.join(results_dir, 'subject_order_*.txt'))
subjects = np.loadtxt(subjects_fname, dtype=str).tolist()

In [ ]:
# Loop through subjects and load pre-training data
files = []
    
for s, subject in enumerate(subjects):
    fname = recent_fname(os.path.join(proc_dir, subject, f'*_{subject}_prepped-raw.fif'))
    if not os.path.exists(fname):
        print(f'No prepped data for {subject}')
        pass
    else:
        files.append(fname)
    
# load data
data = Data(files, sampling_frequency=250)


print(data)

# trim data
trimmed_data = data.trim_time_series(n_embeddings=15, sequence_length=100)

# get alpha
alpha = pickle.load(open(os.path.join(model_dir, "inf_params", "alp_rw.pkl"), "rb"))

In [ ]:
# Calculate regression spectra for each mode and subject (will take a few minutes)
f, psd, coh, w = spectral.regression_spectra(
    data=trimmed_data,
    alpha=alpha,
    sampling_frequency=250,
    frequency_range=[1, 45],
    window_length=1000,
    step_size=20,
    n_sub_windows=8,
    return_coef_int=True,
    return_weights=True,
    n_jobs=16,
)

# re-scale psd
psd_rs = spectral.rescale_regression_coefs(psd, alpha, window_length=1000, 
                                           step_size=20, n_sub_windows=8)

In [ ]:
# Save spectra to disk
if writeOK:
    os.makedirs(op.join(model_dir, "spectra"), exist_ok=True)
    np.save(op.join(model_dir, "spectra", "f.npy"), f)
    np.save(op.join(model_dir, "spectra", "psd.npy"), psd)
    np.save(op.join(model_dir, "spectra", "psd_rs.npy"), psd_rs)
    np.save(op.join(model_dir, "spectra", "coh.npy"), coh)
    np.save(op.join(model_dir, "spectra", "w.npy"), w)